# K-Means Experiments (PySpark)

This notebook runs clustering experiments on multiple datasets
using a manual K-Means implementation.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.utils import create_spark, load_data, normalize_rdd
from src.kmeans import run_kmeans
from src.evaluation import compute_calinski, compute_ari

import numpy as np
import pandas as pd

In [ ]:
spark = create_spark()

In [ ]:
def run_experiments(path):

    rdd, labels = load_data(spark, path)
    rdd = normalize_rdd(spark, rdd)

    results = []

    for k in [2, 3, 4, 5, 6]:
        ch_scores = []
        ari_scores = []

        for _ in range(10):
            centroids = run_kmeans(rdd, k)
            ch_scores.append(compute_calinski(rdd, centroids))
            ari_scores.append(compute_ari(labels, rdd, centroids))

        results.append({
            "k": k,
            "ch_mean": np.mean(ch_scores),
            "ch_std": np.std(ch_scores),
            "ari_mean": np.mean(ari_scores),
            "ari_std": np.std(ari_scores)
        })

    return pd.DataFrame(results)

In [ ]:
df_iris = run_experiments("../data/iris.csv")
df_glass = run_experiments("../data/glass.csv")
df_parkinsons = run_experiments("../data/parkinsons.csv")

df_iris, df_glass, df_parkinsons